# Data Validation

Data validation is the process of ensuring that data meets quality standards and is suitable for analysis or model training. It includes checking for completeness, consistency, accuracy, and conformity to expected formats.

## Why Validate Data?
- Catch errors early in the pipeline
- Prevent corrupted data from affecting models
- Ensure reproducibility
- Maintain data integrity

## Validation Types:
1. **Type Validation** - Check data types
2. **Range Validation** - Verify values within expected bounds
3. **Format Validation** - Ensure proper structure
4. **Completeness** - Check for missing values
5. **Consistency** - Verify logical consistency
6. **Uniqueness** - Check for duplicates

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('/Users/thameem/Desktop/thameem/Machine Learning/Data/height-weight.csv')
print("Dataset loaded:")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

## 1. Type Validation

Verify that columns have the correct data types.

In [ ]:
# Type Validation
print("Data Types:")
print(df.dtypes)

# Check if numeric columns are actually numeric
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"\nNumeric columns: {list(numeric_cols)}")

# Validate types
expected_types = {
    'Gender': 'object',
    'Height': 'float64',
    'Weight': 'float64'
}

print("\nType Validation:")
for col, expected_type in expected_types.items():
    if col in df.columns:
        actual_type = str(df[col].dtype)
        is_valid = actual_type == expected_type
        print(f"{col}: Expected {expected_type}, Got {actual_type} - {'✓' if is_valid else '✗'}")

## 2. Range Validation

Check if numerical values fall within expected ranges.

In [ ]:
# Range Validation
print("Range Validation:")

# Define valid ranges
valid_ranges = {
    'Height': (100, 250),  # cm
    'Weight': (20, 300)    # kg
}

for col, (min_val, max_val) in valid_ranges.items():
    if col in df.columns:
        out_of_range = ((df[col] < min_val) | (df[col] > max_val)).sum()
        print(f"{col}: Range [{min_val}, {max_val}]")
        print(f"  Valid values: {len(df) - out_of_range}")
        print(f"  Out of range: {out_of_range}")
        if out_of_range > 0:
            print(f"  Examples: {df[out_of_range > 0][col].values[:5]}")

## 3. Completeness Validation

Check for missing or null values.

In [ ]:
# Completeness Validation
print("\nCompleteness Validation:")
print(f"Total rows: {len(df)}")

missing_values = df.isnull().sum()
print(f"\nMissing values per column:")
print(missing_values)

missing_percentage = (df.isnull().sum() / len(df)) * 100
print(f"\nMissing values (%):")
for col, pct in missing_percentage.items():
    if pct > 0:
        print(f"  {col}: {pct:.2f}%")
        
if missing_percentage.sum() == 0:
    print("  ✓ No missing values found")

## 4. Uniqueness Validation

Detect duplicate rows and check for unexpected duplicates.

In [ ]:
# Uniqueness Validation
print("\nUniqueness Validation:")

# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows}")

if duplicate_rows > 0:
    print(f"Percentage of duplicates: {(duplicate_rows / len(df)) * 100:.2f}%")
else:
    print("✓ No duplicate rows found")

# Check for duplicate values in key columns
if 'Gender' in df.columns:
    print(f"\nUnique Gender values: {df['Gender'].nunique()}")
    print(df['Gender'].value_counts())

## 5. Consistency Validation

Verify logical consistency in the data.

In [ ]:
# Consistency Validation
print("\nConsistency Validation:")

# Check for logical relationships
# Example: BMI should be Weight / (Height in meters)^2
if 'Height' in df.columns and 'Weight' in df.columns:
    bmi = df['Weight'] / ((df['Height'] / 100) ** 2)
    
    # BMI should be between 10 and 60 for valid human data
    valid_bmi = ((bmi >= 10) & (bmi <= 60)).sum()
    invalid_bmi = ((bmi < 10) | (bmi > 60)).sum()
    
    print(f"Calculated BMI consistency:")
    print(f"  Valid BMI (10-60): {valid_bmi}")
    print(f"  Invalid BMI: {invalid_bmi}")
    
    if invalid_bmi == 0:
        print("  ✓ All BMI values are consistent")

## 6. Data Quality Report

Generate a comprehensive validation report.

In [ ]:
# Comprehensive Data Quality Report
print("\n" + "="*50)
print("DATA QUALITY REPORT")
print("="*50)

report = {
    'Total Rows': len(df),
    'Total Columns': len(df.columns),
    'Duplicate Rows': df.duplicated().sum(),
    'Missing Values': df.isnull().sum().sum(),
    'Data Completeness (%)': (1 - (df.isnull().sum().sum() / (len(df) * len(df.columns)))) * 100
}

for key, value in report.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

# Overall quality score
quality_score = 100
if report['Missing Values'] > 0:
    quality_score -= 10
if report['Duplicate Rows'] > 0:
    quality_score -= 5

print(f"\nOverall Data Quality Score: {quality_score}/100")